# Multi-view reconstruction method benchmark

## 0. Import

In [ ]:
%matplotlib inline
import time
import openalea.phenomenal.data as phm_data
import openalea.phenomenal.object as phm_obj
import openalea.phenomenal.multi_view_reconstruction as phm_mvr
import openalea.phenomenal.display as phm_display
import openalea.phenomenal.display.notebook as phm_display_notebook
from openalea.phenotyping_data.fetch import fetch_all_data

## 1. Load Data

In [ ]:
plant_number = 2  # Available : 1, 2, 3, 4 or 5
data_dir = fetch_all_data(f"plant_{plant_number}")

In [ ]:
bin_images = phm_data.bin_images(data_dir)
calibrations = phm_data.calibrations(data_dir)
image_views = dict()
for id_camera in bin_images:
    for angle in bin_images[id_camera]:
        name = f'{id_camera}_{angle}'
        projection = calibrations[id_camera].get_projection(angle)
        image_views[name] = phm_obj.ImageView(
                bin_images[id_camera][angle],
                projection)

In [ ]:
btimes, bvoxels, bmetrics = {}, {}, {} # best choice of each step to be ploted at the end

## 2. Benchmark voxel size

In [ ]:
times, voxels, metrics = {}, {}, {}
voxels_sizes = [16, 8, 4, 2, 1]

In [ ]:
for voxels_size in voxels_sizes:
    print(voxels_size)
    start = time.time()
    voxel_grid = phm_mvr.reconstruction_3d(
        image_views, voxels_size=voxels_size, clear_outside='side')
    end = time.time()
    times[voxels_size] = end - start
    voxels[voxels_size] = voxel_grid
    metrics[voxels_size] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

In [ ]:
phm_display.plot_reconstruction_dashboard(times, metrics)

In [ ]:
# size 4 is a good compromise
voxels_size = 4 
method = 'rigid'
btimes[method] = times[4]
bvoxels[method] = voxels[4]
bmetrics[method] = metrics[4]

## 2 Benchmark tolerant reconstruction

In [ ]:
times, voxels, metrics = {}, {}, {}
error_tolerance = [0, 1, 2, 3, 4]

In [ ]:
for tol in error_tolerance:
    print(tol)
    start = time.time()
    voxel_grid = phm_mvr.reconstruction_3d(
        image_views, voxels_size=voxels_size, error_tolerance=tol, clear_outside='side')
    end = time.time()
    times[tol] = end - start
    voxels[tol] = voxel_grid
    metrics[tol] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

In [ ]:
binded=phm_obj.bind_grids(list(voxels.values()), -1300)
phm_display_notebook.show_voxel_grid(binded, size=1)

In [ ]:
phm_display.plot_reconstruction_dashboard(times, metrics)

In [ ]:
# tolerance 1 has a slightly better recall/precision/compromise
method = 'tolerant'
btimes[method] = times[1]
bvoxels[method] = voxels[1]
bmetrics[method] = metrics[1]

## 3 Benchmark multi-tolerant reconstruction

In [ ]:
times, voxels, metrics = {}, {}, {}
error_tolerance = [0, -1, -2, -3, -4]

In [ ]:
for tol in error_tolerance:
    print(tol)
    start = time.time()
    voxel_grid = phm_mvr.reconstruction_3d(
    image_views, voxels_size=voxels_size, error_tolerance=tol, clear_outside='side')
    end = time.time()
    times[tol] = end - start
    voxels[tol] = voxel_grid
    metrics[tol] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

In [ ]:
phm_display.plot_reconstruction_dashboard(times, metrics)

In [ ]:
# tolerance -2 has a better recall/precision compromise
method = 'multitolerant'
btimes[method] = times[-2]
bvoxels[method] = voxels[-2]
bmetrics[method] = metrics[-2]

## 5 Comparison of reconstruction methods

### 5.1 Add rigid + 'best view' reconstruction

In [ ]:
method = 'best_view'
error_tolerance = 1
start = time.time()
best_angle = phm_mvr.find_best_angle(bin_images["side"])
image_ref = f'side_{best_angle}'
voxel_grid = phm_mvr.reconstruction_3d_neighbours(
    image_views, voxels_size=voxels_size, error_tolerance=error_tolerance, clear_outside='side', reference_views=image_ref
)
end = time.time()
btimes[method] = end - start
bvoxels[method] = voxel_grid
bmetrics[method] = phm_mvr.reconstruction_metrics(voxel_grid, image_views)

### 5.2 Quantitative comparison

In [ ]:
phm_display.plot_reconstruction_dashboard(btimes, bmetrics)

### 5.3 Visual comparison

In [ ]:
binded=phm_obj.bind_grids(list(bvoxels.values()), -1300)
phm_display_notebook.show_voxel_grid(binded, size=1)